# 00 – Projektüberblick

**Projekt:** WealthScope AI 1.0
**Methode:** QUA³CK · reproduzierbarer Out-of-Time-Benchmark
**Hinweis:** Wissenschaftlicher Prototyp, keine Anlageberatung.

## Zielbild

WealthScope AI verbindet historische US-Aktienmarktdaten mit technischer
Analyse, einem ehrlichen ML-Benchmark, Risikoplanung und Wissenstransfer.
Der Mehrwert liegt in **Nachvollziehbarkeit**, nicht in einem Renditeversprechen.

Die Version 1.0 stellt drei Perspektiven nebeneinander:

1. **KI damals:** Baseline, Logistische Regression, Entscheidungsbaum und SVM.
2. **KI im klassischen ML:** Random Forest, Diagnostik und Erklärbarkeit.
3. **KI heute:** ein erklärender Assistent, klar getrennt vom Prognosemodell.

## Lernziele

Nach diesem Notebook könnt ihr:

- die acht Notebooks der Reihe nach einordnen
- nachvollziehen, welches Artefakt welche Kennzahl verbindlich festlegt
- das zentrale Ergebnis des Projekts in einem Satz benennen

## Leseanleitung

Jedes Notebook beantwortet genau eine Frage. Wer nur das Ergebnis sucht,
springt zu **05**; wer die Methodik prüfen will, liest **03** und **04**.

| Notebook | Beantwortet | Kernbefund |
|---|---|---|
| 00 Überblick | Worum geht es? | Ehrlichkeit vor Renditeversprechen |
| 01 Question | Was wird gefragt, was widerlegt es? | H1 ist falsifizierbar formuliert |
| 02 Understanding | Welche Daten liegen vor? | Strukturelle Fehlwerte, instabile Klassenlage |
| 03 Feature Engineering | Was darf ein Feature sein? | Der Prognosezeitpunkt entscheidet |
| 04 Modeling | Welches Modell trägt? | Keines schlägt die Baseline; Leakage täuscht |
| 05 Conclude | Was bleibt? | H1 falsifiziert, mit Konfidenzintervall |
| 06 Knowledge | Wie wird es zugänglich? | Streamlit, Export, Lernstudio |
| 07 NewsAPI | Was macht die KI-Ebene? | Erklären, nicht prognostizieren |

> **Arbeitsregel des Projekts:** Jede Zahl in App, Ausarbeitung, Poster und
> Notebooks stammt aus `models/*.json`. Wo eine Zahl von Hand getippt wurde, ist
> sie erfahrungsgemäß nach der ersten Änderung falsch.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "wealthscope_features.parquet"
DIAGNOSTICS_PATH = PROJECT_ROOT / "models" / "diagnostics.json"
EXPERIMENTS_PATH = PROJECT_ROOT / "models" / "validation_experiments.json"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Datensatz fehlt: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(f"Daten: {len(df):,} Zeilen × {len(df.columns)} Spalten")
display(df.head(3))

Daten: 192,119 Zeilen × 27 Spalten


,date,open,high,low,close,volume,ticker,asset_type,source_file,daily_return,...,ma_200_distance,volatility_20d,rolling_high,drawdown,future_return_20d,target_20d,open_interest,volatility_60d,rolling_low_60,rolling_high_60
0,1985-06-21,0.25742,0.26381,0.25742,0.25742,46333854,AAPL,Stock,Stocks/aapl.us.txt,0.020374,...,-0.331288,0.040521,0.48791,-0.472403,0.044635,1.0,NaN,NaN,NaN,NaN
1,1985-06-24,0.27535,0.27920,0.27535,0.27535,57384755,AAPL,Stock,Stocks/aapl.us.txt,0.069653,...,-0.283328,0.040440,0.48791,-0.435654,-0.041910,0.0,NaN,NaN,NaN,NaN
2,1985-06-25,0.27920,0.28556,0.27920,0.27920,81966629,AAPL,Stock,Stocks/aapl.us.txt,0.013982,...,-0.271961,0.037120,0.48791,-0.427763,-0.073424,0.0,NaN,NaN,NaN,NaN


In [2]:
overview = pd.Series({
    "Zeilen": len(df),
    "Ticker": df["ticker"].nunique(),
    "Zeitraum von": df["date"].min().date(),
    "Zeitraum bis": df["date"].max().date(),
    "Zielvariable vorhanden": "target_20d" in df.columns,
    "Diagnostik vorhanden": DIAGNOSTICS_PATH.exists(),
})
overview.to_frame("Wert")

,Wert
Zeilen,192119
Ticker,26
Zeitraum von,1962-01-02
Zeitraum bis,2017-11-10
Zielvariable vorhanden,True
Diagnostik vorhanden,True


In [3]:
qua3ck = pd.DataFrame([
    ["Q", "Question", "Leitfrage und Hypothesen"],
    ["U", "Understanding", "Datenprofil, Fehlwerte, Klassen und Zeit"],
    ["A", "Analytics", "Returns, Trend- und Risikofeatures"],
    ["A", "Algorithm", "Fünf Klassifikatoren auf identischen Fenstern"],
    ["A", "Adaption", "Pipeline, Purge, Walk-forward und Hyperparameter"],
    ["C", "Conclude", "Metriken, Grenzen und Hypothesenbewertung"],
    ["K", "Knowledge", "Streamlit, Export, Lernstudio und Dokumentation"],
], columns=["Phase", "Name", "WealthScope-Artefakt"])
qua3ck

,Phase,Name,WealthScope-Artefakt
0,Q,Question,Leitfrage und Hypothesen
1,U,Understanding,"Datenprofil, Fehlwerte, Klassen und Zeit"
2,A,Analytics,"Returns, Trend- und Risikofeatures"
3,A,Algorithm,Fünf Klassifikatoren auf identischen Fenstern
4,A,Adaption,"Pipeline, Purge, Walk-forward und Hyperparameter"
5,C,Conclude,"Metriken, Grenzen und Hypothesenbewertung"
6,K,Knowledge,"Streamlit, Export, Lernstudio und Dokumentation"


## Reproduzierbarkeit

Die Notebooks erklären und prüfen die Artefakte. Die verbindliche Trainingslogik
liegt in `scripts/train_and_diagnose.py`; Kennzahlen werden aus
`models/diagnostics.json` gelesen. So widersprechen sich App, Ausarbeitung und
Notebooks nicht.

## Management-Checkpoint

WealthScope AI liefert **kein** Handelssignal. Der Ertrag der Arbeit ist eine
Messung: Wie viel verwertbare Information tragen rein kursbasierte technische
Indikatoren? Antwort — nahezu keine, belegt an 190.527 Beobachtungen mit elf
Jahren unangetastetem Testzeitraum. Wer das Projekt in einem Satz zusammenfasst,
nennt dieses Ergebnis, nicht die App.